# TT backend 完整训练测速

本 Notebook 专门比较不同 TT backend 的完整 CausalLM 训练时间，不执行 WikiText 质量评测。每个 backend 都从同一份 canonical TT cores 开始，在独立加载的 Qwen 模型上使用相同固定合成 token。

计时范围包含 forward、backward、activation checkpoint 重计算、梯度裁剪和 AdamW step；模型加载与 TT 安装时间单独记录。

In [1]:
import gc
import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/xls/workspace/projects/qwen3-tn-compression").resolve()
SRC_ROOT = PROJECT_ROOT / "src"
SCRIPT_PATH = PROJECT_ROOT / "scripts/benchmark_tt_training.py"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("Python:", sys.executable)
print("项目：", PROJECT_ROOT)
print("脚本：", SCRIPT_PATH)

Python: /home/xls/appdata/miniforge3/envs/qwen3-tn/bin/python
项目： /mnt/intern7/xls/projects/qwen3-tn-compression
脚本： /mnt/intern7/xls/projects/qwen3-tn-compression/scripts/benchmark_tt_training.py


## 1. 配置

默认 256 tokens、无梯度累积，用于安全、快速地比较 backend。要贴近 Notebook 06 的正式训练，可改为 `SEQUENCE_LENGTH=1024`、`GRADIENT_ACCUMULATION_STEPS=8`；高显存 backend 如果 OOM，会将错误写入结果 JSON 并继续测试其他 backend。

In [2]:
MODEL_PATH = Path("/infini-data/Qwen3-8B")
CHECKPOINT_CANDIDATES = (
    PROJECT_ROOT / "artifacts/tt_finetune_wikitext_layer_0_5_quick/backends/native/final/tt_modules",
    PROJECT_ROOT / "artifacts/tt_finetune_wikitext_layer_0_5_quick/final/tt_modules",
)
TT_CHECKPOINT_ROOT = next(
    (path for path in CHECKPOINT_CANDIDATES if path.is_dir()),
    CHECKPOINT_CANDIDATES[0],
)
BACKENDS = ("native", "tensorly_torch")
DEVICE = "cuda:0"

SEQUENCE_LENGTH = 256
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
WARMUP_STEPS = 1
MEASURED_STEPS = 5
TOKEN_CHUNK_SIZE = 8
LEARNING_RATE = 1e-5

BF16_AUTOCAST = True
GRADIENT_CHECKPOINTING = True
TT_ACTIVATION_CHECKPOINTING = True
FAIL_FAST = False
RUN_BENCHMARK = True
OUTPUT_PATH = PROJECT_ROOT / "results/tt_training_backend_benchmark.json"

print("模型：", MODEL_PATH)
print("TT cores：", TT_CHECKPOINT_ROOT)
print("Backends：", BACKENDS)
print("输出：", OUTPUT_PATH)

模型： /infini-data/Qwen3-8B
TT cores： /mnt/intern7/xls/projects/qwen3-tn-compression/artifacts/tt_finetune_wikitext_layer_0_5_quick/final/tt_modules
Backends： ('native', 'tensorly_torch')
输出： /mnt/intern7/xls/projects/qwen3-tn-compression/results/tt_training_backend_benchmark.json


## 2. 运行完整训练 benchmark

脚本在子进程中依次加载和释放每个 backend，避免两个 8B 模型同时驻留。运行期间不要在另一个 Kernel 中保留同一张 GPU 上的大模型。

In [3]:
if RUN_BENCHMARK:
    if not MODEL_PATH.is_dir():
        raise FileNotFoundError(f"模型目录不存在：{MODEL_PATH}")
    if not TT_CHECKPOINT_ROOT.is_dir():
        raise FileNotFoundError(
            f"TT checkpoint 不存在：{TT_CHECKPOINT_ROOT}。先完成一次 TT 分解或训练。"
        )
    environment = os.environ.copy()
    existing_pythonpath = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = (
        str(SRC_ROOT)
        if not existing_pythonpath
        else str(SRC_ROOT) + os.pathsep + existing_pythonpath
    )
    command = [
        sys.executable,
        str(SCRIPT_PATH),
        "--model-path", str(MODEL_PATH),
        "--tt-checkpoint-root", str(TT_CHECKPOINT_ROOT),
        "--backends", *BACKENDS,
        "--device", DEVICE,
        "--sequence-length", str(SEQUENCE_LENGTH),
        "--batch-size", str(BATCH_SIZE),
        "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
        "--warmup-steps", str(WARMUP_STEPS),
        "--steps", str(MEASURED_STEPS),
        "--token-chunk-size", str(TOKEN_CHUNK_SIZE),
        "--learning-rate", str(LEARNING_RATE),
        "--bf16-autocast" if BF16_AUTOCAST else "--no-bf16-autocast",
        "--gradient-checkpointing" if GRADIENT_CHECKPOINTING else "--no-gradient-checkpointing",
        "--tt-activation-checkpointing" if TT_ACTIVATION_CHECKPOINTING else "--no-tt-activation-checkpointing",
        "--output", str(OUTPUT_PATH),
    ]
    if FAIL_FAST:
        command.append("--fail-fast")
    print("运行：", " ".join(command))
    subprocess.run(command, check=True, env=environment)
    gc.collect()
else:
    print("RUN_BENCHMARK=False：跳过测速，下一节将读取已有 JSON。")

运行： /home/xls/appdata/miniforge3/envs/qwen3-tn/bin/python /mnt/intern7/xls/projects/qwen3-tn-compression/scripts/benchmark_tt_training.py --model-path /infini-data/Qwen3-8B --tt-checkpoint-root /mnt/intern7/xls/projects/qwen3-tn-compression/artifacts/tt_finetune_wikitext_layer_0_5_quick/final/tt_modules --backends native tensorly_torch --device cuda:0 --sequence-length 256 --batch-size 1 --gradient-accumulation-steps 1 --warmup-steps 1 --steps 5 --token-chunk-size 8 --learning-rate 1e-05 --bf16-autocast --gradient-checkpointing --tt-activation-checkpointing --output /mnt/intern7/xls/projects/qwen3-tn-compression/results/tt_training_backend_benchmark.json

benchmarking backend=native


Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 126.08it/s]


native: step=1/5 loss=12.770004 time=2598.298 ms
native: step=2/5 loss=12.419436 time=2610.146 ms
native: step=3/5 loss=12.176588 time=2612.497 ms
native: step=4/5 loss=12.002700 time=2618.021 ms
native: step=5/5 loss=11.852313 time=2754.133 ms
native: median optimizer step=2612.497 ms, tokens/s=97.02

benchmarking backend=tensorly_torch


Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 96.03it/s]


tensorly_torch: step=1/5 loss=12.778016 time=3350.651 ms
tensorly_torch: step=2/5 loss=12.414634 time=3353.093 ms
tensorly_torch: step=3/5 loss=12.167253 time=3362.247 ms
tensorly_torch: step=4/5 loss=11.996008 time=3362.643 ms
tensorly_torch: step=5/5 loss=11.813812 time=3372.113 ms
tensorly_torch: median optimizer step=3362.247 ms, tokens/s=76.19
saved: /mnt/intern7/xls/projects/qwen3-tn-compression/results/tt_training_backend_benchmark.json


## 3. 查看结果

`median_ms` 越小越好，`tokens_per_second` 越大越好。`peak_allocated_gib` 是完整模型训练阶段的 CUDA 峰值显存。

In [4]:
if not OUTPUT_PATH.is_file():
    raise FileNotFoundError(f"结果文件不存在：{OUTPUT_PATH}")
benchmark = json.loads(OUTPUT_PATH.read_text(encoding="utf-8"))
successful = {}
print(
    f"{'backend':<18} {'status':<8} {'median ms':>12} {'mean ms':>12} "
    f"{'tokens/s':>12} {'peak GiB':>12} {'load s':>10}"
)
print("-" * 90)
for backend in benchmark['config']['backends']:
    result = benchmark['results'][backend]
    if result.get('status') != 'ok':
        print(f"{backend:<18} {'error':<8} {result['error_type']}: {result['error']}")
        continue
    timing = result['optimizer_step']
    peak = timing['peak_allocated_bytes']
    peak_gib = peak / 1024**3 if peak is not None else float('nan')
    successful[backend] = timing
    print(
        f"{backend:<18} {'ok':<8} {timing['median_ms']:>12.3f} "
        f"{timing['mean_ms']:>12.3f} {timing['tokens_per_second']:>12.2f} "
        f"{peak_gib:>12.2f} {result['model_load_seconds']:>10.2f}"
    )

if successful:
    reference_name = next(
        name for name in benchmark['config']['backends'] if name in successful
    )
    reference_ms = successful[reference_name]['median_ms']
    print(f"\n以 {reference_name} 为基准：")
    for backend, timing in successful.items():
        speedup = reference_ms / timing['median_ms']
        print(f"  {backend}: {speedup:.3f}x")
print("\n完整 JSON：", OUTPUT_PATH)

backend            status      median ms      mean ms     tokens/s     peak GiB     load s
------------------------------------------------------------------------------------------
native             ok           2612.497     2638.619        97.02        26.89       6.62
tensorly_torch     ok           3362.247     3360.149        76.19        43.23       5.78

以 native 为基准：
  native: 1.000x
  tensorly_torch: 0.777x

完整 JSON： /mnt/intern7/xls/projects/qwen3-tn-compression/results/tt_training_backend_benchmark.json
